# Visualisation of LunarLender runs

## Import libraries

In [2]:
import gymnasium as gym
import mlflow
import numpy as np
import pandas as pd
import os
import io
import glob
import matplotlib.pyplot as plt
import time
import sys
from matplotlib.figure import Figure
from IPython.display import clear_output, display, HTML, display_html, Video
from PIL import Image
from gymnasium.spaces import Box, Discrete
from gymnasium.wrappers import RecordVideo
from stable_baselines3 import A2C, PPO, SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, BaseCallback
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv
from stable_baselines3.common.logger import HumanOutputFormat, KVWriter, Logger
from collections import deque
from typing import Any, Dict, Tuple, Union
from contextlib import redirect_stdout
from agent import train_from_config, train_from_dict


## Create and display episode recording function

In [3]:

def record_episode(model, run_artifact_dir, env=None, episode_num=None):
    """Record a single episode and return reward and length.
    If env is provided, uses it instead of creating a new one.
    If episode_num is provided, uses it in the video name."""
    recording_dir = f"{run_artifact_dir}/recordings"
    os.makedirs(recording_dir, exist_ok=True)

    if env is None:
        # Create new environment if none provided
        env = gym.make(ExpConf.ENV_ID, render_mode="rgb_array")

    with open(os.devnull, 'w') as f, redirect_stdout(f):
        # Wrap with RecordVideo
        record_env = RecordVideo(
            env=env,
            video_folder=recording_dir,
            episode_trigger=lambda episode_id: True,  # Record the episode
            name_prefix=f"lunar_lander_episode_{episode_num if episode_num is not None else 'test'}"
        )

        # Run and record a single episode
        obs, _ = record_env.reset()
        done = False
        episode_reward = 0
        episode_length = 0

        while not done:
            action, _states = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = record_env.step(action)
            episode_reward += reward
            episode_length += 1
            done = terminated or truncated
        record_env.close()

    return episode_reward, episode_length, recording_dir

## Run evaluation from best_model_dir and record all episodes

In [4]:
# Select run directory
run_dir = f"/app/data/models/mlflow_experiments/326026350238866536/1aee3015e2314ae8948e9ce8b3bab396"
model_dir = f"{run_dir}/artifacts/best_model"
params_dir = f"{run_dir}/params"

with open(os.path.join(params_dir, "MODEL_ID"), "r") as f:
    MODEL_ID = f.read().strip()

with open(os.path.join(params_dir, "ENV_ID"), "r") as f:
    ENV_ID = f.read().strip()

# Create environment with render mode for recording
eval_env = gym.make(ENV_ID, render_mode="rgb_array")
eval_env = Monitor(eval_env)

# Load model
if MODEL_ID == "PPO":
    model = PPO.load(model_dir)
elif MODEL_ID == "A2C":
    model = A2C.load(model_dir)
elif MODEL_ID == "SAC":
    model = SAC.load(model_dir)
else:
    raise ValueError(f"Unsupported model type: {MODEL_ID}")

# Lists to store episode metrics
episode_rewards = []
episode_lengths = []
n_eval_episodes = 10

# Evaluate and record each episode
for episode in range(n_eval_episodes):
    reward, length, _ = record_episode(model, run_dir, env=eval_env, episode_num=episode)
    episode_rewards.append(reward)
    episode_lengths.append(length)

# Create DataFrame with metrics
metrics_df = pd.DataFrame({
    'Episode': range(1, n_eval_episodes + 1),
    'Reward': episode_rewards,
    'Length': episode_lengths
})

# Add summary statistics
summary_df = pd.DataFrame({
    'Episode': ['Mean', 'Std'],
    'Reward': [metrics_df['Reward'].mean(), metrics_df['Reward'].std()],
    'Length': [metrics_df['Length'].mean(), metrics_df['Length'].std()]
})

# Display metrics table with summary
print("\nEvaluation Results:")
print(metrics_df.to_string(index=False))
print("\nSummary Statistics:")
print(summary_df.to_string(index=False))


/usr/local/lib/python3.10/dist-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /app/data/models/mlflow_experiments/326026350238866536/1aee3015e2314ae8948e9ce8b3bab396/recordings folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.10/dist-packages/gymnasium/wrappers/record_video.py:94: UserWarning: WARN: Overwriting existing videos at /app/data/models/mlflow_experiments/326026350238866536/1aee3015e2314ae8948e9ce8b3bab396/recordings folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
                                                                          


Evaluation Results:
 Episode     Reward  Length
       1 299.027813     264
       2 266.802033     290
       3 255.614383     296
       4 211.665524     428
       5 265.014433     327
       6 218.205862     361
       7 236.549447     664
       8 287.960700     276
       9 285.156866     288
      10 242.631319     307

Summary Statistics:
Episode     Reward     Length
   Mean 256.862838 350.100000
    Std  29.560931 120.374831


# Display the selected episode's recording

In [6]:
episode_to_show = 8

video_path = glob.glob(f"{run_dir}/recordings/*episode_{episode_to_show-1}*.mp4")[0]
display(Video(video_path, embed=True, width=800, height=600))
print(f"Episode {episode_to_show} reward: {episode_rewards[episode_to_show-1]:.2f}")
print(f"Episode {episode_to_show} length: {episode_lengths[episode_to_show-1]}")


Episode 8 reward: 287.96
Episode 8 length: 276


In [8]:
nvidia-smi

NameError: name 'nvidia' is not defined